In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [11]:
sns.set(style="whitegrid") 
df = pd.read_parquet('../data/hotels.parquet' , engine='fastparquet') 
print(f"Original dataset shape: {df.shape}")
display(df.head(5))
# df.iloc[100000:100003] # Note: In Python slicing [a:b], the 'b' index is exclusive (not included)

Original dataset shape: (119390, 17)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,0,0,0,C,C
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,0,0,0,C,C
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,0,0,0,A,C
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,0,0,0,A,A
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,0,0,0,A,A


In [19]:
missing_values = df.isnull().sum()
missing_ratio = df.isnull().mean() * 100
missing_summary = pd.DataFrame({'Missing Count': missing_values,'Missing (%)': missing_ratio})

# Keep only columns with missing values and sort by percentage descending
missing_summary = missing_summary[missing_summary['Missing Count'] > 0].sort_values(by='Missing (%)', ascending=False)

if missing_summary.empty:
    print("No missing values found in the dataset.")
else:
    print("Missing values summary (all columns with missing data):")
    for idx, row in missing_summary.iterrows():
        print(f"Column '{idx}': {int(row['Missing Count'])} missing values ({row['Missing (%)']:.4f}%)")

Missing values summary (all columns with missing data):
Column 'children': 4 missing values (0.0034%)


In [20]:
df['children'] = df['children'].fillna(0).astype(int)
print(f"Filled missing values in 'children' column with 0.")

Filled missing values in 'children' column with 0.


In [21]:
df['total_guests'] = df['adults'] + df['children'] + df['babies']
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
print("Calculated 'total_guests' and 'total_nights' for each booking.")

Calculated 'total_guests' and 'total_nights' for each booking.


In [22]:
rows_before = df.shape[0]
df_clean = df[df['adults'] > 0].copy()
rows_after = df_clean.shape[0]
print(f"Removed {rows_before - rows_after} rows with zero adults.")

rows_before = df_clean.shape[0]
df_clean = df_clean[df_clean['total_nights'] > 0].copy()
rows_after = df_clean.shape[0]
print(f"Removed {rows_before - rows_after} rows with zero total nights.")

rows_before = df_clean.shape[0]
df_clean.drop_duplicates(inplace=True)
rows_after = df_clean.shape[0]
print(f"Removed {rows_before - rows_after} duplicate rows.")

print(f"\nOriginal dataset rows: {df.shape[0]}")
print(f"Cleaned dataset rows: {df_clean.shape[0]}")
print(f"\nTotal rows removed during cleaning: {len(df) - len(df_clean)}")

Removed 403 rows with zero adults.
Removed 645 rows with zero total nights.
Removed 39814 duplicate rows.

Original dataset rows: 119390
Cleaned dataset rows: 78528

Total rows removed during cleaning: 40862


In [23]:
df_clean.to_parquet(
    "../data/hotels_clean.parquet",
    engine="fastparquet",
    index=False
)

print("Clean dataset saved to ../data/hotels_clean.parquet")

Clean dataset saved to ../data/hotels_clean.parquet
